# ELF–TBCC GPU Grid Search

Before opening this notebook, manually clone the intended branch in Colab:

```bash
git clone --branch dev-spectrum-codex https://github.com/UCLA-Communications-Systems-Lab/tbcc-decoder-test.git /content/tbcc-decoder-test
```

This notebook does not clone or pull repositories. Edit the configuration cell below, then compile, run, merge, and report through `gridsearch.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import yaml

# Edit these code parameters for each grid search.
K = 11
ELF_MEMORY = 4                 # m
TBCC_MEMORY = 5                # nu
RATE_DENOMINATOR = 2           # rate 1/2 or 1/3
TARGET_EBNO_DB = 6.5
ELF_POLYNOMIAL = None          # None searches all ELF polynomials
GEN_POLYS = None               # None searches all valid TBCC generators

REPO_DIR = Path('/content/tbcc-decoder-test')
SPECTRUM_DIR = REPO_DIR / 'distance_spectrum_computation'
RESULTS_DIR = Path('/content/drive/MyDrive/TBCC_Results')
BATCH_INDEX = 0
BATCH_SIZE = 500
CYCLIC_ONLY = False
LABEL = 'gridsearch'

N_ELF = K + ELF_MEMORY
N_TBCC = RATE_DENOMINATOR * N_ELF
config = {
    'bch_config': {'K': K, 'N': N_ELF, 'M': ELF_MEMORY},
    'tbcc_config': {'K': N_ELF, 'N': N_TBCC, 'V': TBCC_MEMORY},
    'target_EbNo_dB': TARGET_EBNO_DB,
}
if ELF_POLYNOMIAL is not None:
    config['bch_config']['polynomial'] = ELF_POLYNOMIAL
if GEN_POLYS is not None:
    config['tbcc_config']['gen_polys'] = GEN_POLYS

CONFIG_PATH = SPECTRUM_DIR / 'config' / f'k{K}n{N_TBCC}v{TBCC_MEMORY}_{LABEL}.yaml'
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
with CONFIG_PATH.open('w') as config_file:
    yaml.safe_dump(config, config_file, sort_keys=False)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULT_PREFIX = CONFIG_PATH.stem
RESULT_FILE = RESULTS_DIR / f'{RESULT_PREFIX}.h5'  # use *_batchN.h5 or *_merged.h5 when applicable
print(f'Wrote {CONFIG_PATH}')
print(yaml.safe_dump(config, sort_keys=False))

In [ ]:
import subprocess

subprocess.run(['bash', 'foldshift_compile.sh'], cwd=SPECTRUM_DIR / 'compile', check=True)

In [ ]:
command = [
    'python', 'gridsearch.py', 'run', str(CONFIG_PATH),
    '--output-dir', str(RESULTS_DIR),
    '--batch-index', str(BATCH_INDEX),
    '--batch-size', str(BATCH_SIZE),
    '--label', LABEL,
]
if CYCLIC_ONLY:
    command.append('--cyclic')
subprocess.run(command, cwd=SPECTRUM_DIR, check=True)

In [ ]:
# Run this only after producing two or more batch files.
subprocess.run(
    ['python', 'gridsearch.py', 'merge', str(RESULTS_DIR), RESULT_PREFIX],
    cwd=SPECTRUM_DIR,
    check=True,
)

In [ ]:
# Set RESULT_FILE in the configuration cell to the unbatched, batch, or merged result file.
subprocess.run(
    ['python', 'gridsearch.py', 'report', str(RESULT_FILE)],
    cwd=SPECTRUM_DIR,
    check=True,
)